# Chapter 6: Multiple Regression — Further Issues
### Heiss & Brunner (2024)

## Topics
1. Scaling and standardised (beta) coefficients
2. Log functional forms — all 4 types compared
3. Quadratic models and turning points
4. Interaction terms
5. Adjusted R-squared
6. Prediction and prediction intervals

**Format:** One step per cell. Manual first, then statsmodels, then compare.

## Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.formula.api as smf
import wooldridge as woo

In [ ]:
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)
print('Working directory:', os.getcwd())

In [ ]:
df = woo.dataWoo('wage1')
df['lwage'] = np.log(df['wage'])
print('wage1 loaded:', df.shape)
print(df[['wage','lwage','educ','exper','tenure','female']].head(8).to_string())

---
## Section 1 — Scaling and Standardised Coefficients

### WHY standardise?
Different variables have different units (years, dollars, 0/1).
Standardised coefficients allow you to compare the relative importance of each variable.

### Beta coefficient formula
$$\hat{b}_j = \hat{\beta}_j \cdot \frac{s_{x_j}}{s_y}$$

### Example Question
> Which has a larger standardised effect on log wages: education or experience?

### Step 1.1 — Run the baseline regression

In [ ]:
res_base = smf.ols('lwage ~ educ + exper + tenure + female', data=df).fit()
print('Baseline regression: lwage ~ educ + exper + tenure + female')
print(res_base.summary())

### Step 1.2 — Manually standardise all variables

In [ ]:
print('BEFORE standardisation:')
print(df[['lwage','educ','exper','tenure']].describe().round(3).to_string())

In [ ]:
for col in ['lwage','educ','exper','tenure','female']:
    mean_c = df[col].mean()
    std_c  = df[col].std(ddof=1)
    df[f'z_{col}'] = (df[col] - mean_c) / std_c
print('AFTER standardisation (z-scores have mean=0, std=1):')
print(df[['z_lwage','z_educ','z_exper','z_tenure']].describe().round(3).to_string())

### Step 1.3 — Run regression on standardised variables

In [ ]:
res_std = smf.ols('z_lwage ~ z_educ + z_exper + z_tenure + z_female', data=df).fit()
print('Standardised regression coefficients:')
for var in ['z_educ','z_exper','z_tenure','z_female']:
    print(f'  {var:<12}: {res_std.params[var]:.4f}')

### Step 1.4 — Compute beta coefficients manually

In [ ]:
sy = df['lwage'].std(ddof=1)
print('Manual beta coefficients: b_hat * (sx / sy)')
print(f'  sy (std of lwage) = {sy:.4f}')
for var, xname in [('educ','z_educ'),('exper','z_exper'),('tenure','z_tenure'),('female','z_female')]:
    sx     = df[var].std(ddof=1)
    b_hat  = res_base.params[var]
    beta   = b_hat * (sx / sy)
    beta_sm= res_std.params[xname]
    match  = np.isclose(beta, beta_sm)
    print(f'  {var:<8}: b_hat={b_hat:.4f}, sx={sx:.4f}, beta_manual={beta:.4f}, beta_sm={beta_sm:.4f}, match={match}')

### Step 1.5 — Interpret

In [ ]:
print('INTERPRETATION of standardised coefficients:')
for var, xname in [('educ','z_educ'),('exper','z_exper'),('tenure','z_tenure')]:
    b = res_std.params[xname]
    print(f'  {var:<8}: {b:.4f} -> a 1-std-dev increase in {var} is associated with')
    print(f'           a {abs(b):.4f} std-dev {"increase" if b>0 else "decrease"} in log(wage)')

---
## Section 2 — Log Functional Forms

### All 4 models
| Model | Equation | Interpretation |
|-------|----------|----------------|
| Level-level | wage ~ educ | +$β₁/hour per year |
| Log-level | lwage ~ educ | ~β₁×100% per year |
| Level-log | wage ~ log(educ) | +$(β₁/100)% per 1% increase |
| Log-log | lwage ~ log(educ) | β₁% per 1% increase (elasticity) |

### Step 2.1 — Create log(educ) and show before/after

In [ ]:
print('educ values (first 5):', df['educ'].values[:5])
df['leduc'] = np.log(df['educ'].replace(0, np.nan))  # avoid log(0)
print('log(educ) values (first 5):', df['leduc'].values[:5].round(4))
print()
print('Note: log(12) =', round(np.log(12), 4), '  log(16) =', round(np.log(16), 4))

### Step 2.2 — Run all 4 models with statsmodels

In [ ]:
res_ll  = smf.ols('wage  ~ educ',  data=df).fit()   # level-level
res_lg  = smf.ols('lwage ~ educ',  data=df).fit()   # log-level
res_lv  = smf.ols('wage  ~ leduc', data=df).fit()   # level-log
res_lolo= smf.ols('lwage ~ leduc', data=df).fit()   # log-log
print('All 4 models estimated.')

### Step 2.3 — Compare manually computed slopes with statsmodels

In [ ]:
# Verify log-level manually
y_lg = df['lwage'].values
x_ed = df['educ'].values
xd = x_ed - x_ed.mean()
yd = y_lg - y_lg.mean()
b1_lg_manual = np.sum(xd*yd)/np.sum(xd**2)
b1_lg_sm     = res_lg.params['educ']
print('Log-level beta1:')
print(f'  Manual     : {b1_lg_manual:.6f}')
print(f'  statsmodels: {b1_lg_sm:.6f}')
print(f'  Match?     : {np.isclose(b1_lg_manual, b1_lg_sm)}')

### Step 2.4 — Summary table of all 4 models

In [ ]:
models = {'Level-level':'wage~educ','Log-level':'lwage~educ','Level-log':'wage~leduc','Log-log':'lwage~leduc'}
fits = [res_ll, res_lg, res_lv, res_lolo]
print(f'{"Model":<14} {"Dependent":>10} {"Regressor":>10} {"beta1":>10} {"R2":>8} {"Interpretation"}')
print('-'*80)
interp = ['$+{:.3f}/hr per yr educ','~{:.2f}% per yr educ','$+{:.2f} per 1% educ','{:.3f}% per 1% educ']
for (name, formula), fit, fmt in zip(models.items(), fits, interp):
    dep, reg = formula.split('~')
    b1 = fit.params[reg.strip()]
    r2 = fit.rsquared
    print(f'{name:<14} {dep.strip():>10} {reg.strip():>10} {b1:>10.4f} {r2:>8.4f}  {fmt.format(b1)}')

### Step 2.5 — Visualise all 4 models

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
x_line = np.linspace(df['educ'].min(), df['educ'].max(), 200)
x_line_log = np.log(x_line.clip(1))
pairs = [
    (axes[0,0], df['educ'], df['wage'],  res_ll,   x_line, x_line,     'educ',  'wage',  'Level-Level'),
    (axes[0,1], df['educ'], df['lwage'], res_lg,   x_line, x_line,     'educ',  'lwage', 'Log-Level'),
    (axes[1,0], df['educ'], df['wage'],  res_lv,   x_line, x_line_log, 'leduc', 'wage',  'Level-Log'),
    (axes[1,1], df['educ'], df['lwage'], res_lolo, x_line, x_line_log, 'leduc', 'lwage', 'Log-Log'),
]
for ax, xs, ys, res, xl, xl_reg, regname, depname, title in pairs:
    ax.scatter(xs, ys, alpha=0.25, s=15, color='steelblue')
    b0 = res.params['Intercept']
    b1 = res.params[regname]
    ax.plot(xl, b0 + b1*xl_reg, 'r-', lw=2.5, label=f'beta1={b1:.3f}')
    ax.set_title(f'{title}\nR2={res.rsquared:.3f}', fontsize=11, fontweight='bold')
    ax.set_xlabel('educ'); ax.set_ylabel(depname)
    ax.legend(fontsize=8)
plt.suptitle('Four Log Functional Forms', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_ch06_01_log_forms.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Section 3 — Quadratic Models

### WHY quadratic?
Diminishing returns: each extra year of experience raises wages by less as experience grows.

### The model
$$lwage = \beta_0 + \beta_1 exper + \beta_2 exper^2 + \cdots + u$$

### Turning point
$$x^* = -\frac{\hat{\beta}_1}{2\hat{\beta}_2}$$

### Step 3.1 — Create exper² and show before/after

In [ ]:
print('exper (first 5):', df['exper'].values[:5])
df['expersq'] = df['exper'] ** 2
print('expersq (first 5):', df['expersq'].values[:5])
print()
print('Note: expersq = exper^2 captures diminishing returns')

### Step 3.2 — Estimate quadratic model manually

In [ ]:
y_q = df['lwage'].values
X_q = np.column_stack([np.ones(len(df)), df['educ'].values, df['exper'].values, df['expersq'].values, df['tenure'].values, df['female'].values])
beta_q = np.linalg.inv(X_q.T @ X_q) @ (X_q.T @ y_q)
print('Manual OLS estimates (quadratic model):')
for lab, b in zip(['Intercept','educ','exper','expersq','tenure','female'], beta_q):
    print(f'  {lab:<12}: {b:.6f}')

### Step 3.3 — Estimate with statsmodels

In [ ]:
res_quad = smf.ols('lwage ~ educ + exper + expersq + tenure + female', data=df).fit()
print('statsmodels estimates:')
for var in ['Intercept','educ','exper','expersq','tenure','female']:
    print(f'  {var:<12}: {res_quad.params[var]:.6f}')

### Step 3.4 — Compare manual vs statsmodels

In [ ]:
print('COMPARISON:')
for lab, b_m, var in zip(['Intercept','educ','exper','expersq','tenure','female'], beta_q, ['Intercept','educ','exper','expersq','tenure','female']):
    b_sm = res_quad.params[var]
    print(f'  {lab:<12}: manual={b_m:.6f}  sm={b_sm:.6f}  match={np.isclose(b_m, b_sm)}')

### Step 3.5 — Compute the turning point manually

In [ ]:
b_exper  = res_quad.params['exper']
b_expersq= res_quad.params['expersq']
turning_point = -b_exper / (2 * b_expersq)
print('Turning point (experience where returns peak):')
print(f'  x* = -beta_exper / (2*beta_expersq)')
print(f'     = -{b_exper:.4f} / (2*{b_expersq:.6f})')
print(f'     = {turning_point:.1f} years')
print(f'  -> Returns to experience peak at {turning_point:.1f} years, then diminish')
print(f'  -> {100*df["exper"].gt(turning_point).mean():.1f}% of workers in sample exceed this')

### Step 3.6 — Plot the quadratic curve

In [ ]:
b0_q = res_quad.params['Intercept']
b_ed = res_quad.params['educ']
b_ex = res_quad.params['exper']
b_ex2= res_quad.params['expersq']
b_ten= res_quad.params['tenure']
b_fem= res_quad.params['female']
# Hold other vars at their mean, vary exper
mean_educ = df['educ'].mean(); mean_ten = df['tenure'].mean()
exper_range = np.linspace(0, 50, 200)
pred_quad = b0_q + b_ed*mean_educ + b_ex*exper_range + b_ex2*exper_range**2 + b_ten*mean_ten
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(df['exper'], df['lwage'], alpha=0.25, s=15, color='steelblue')
ax.plot(exper_range, pred_quad, 'r-', lw=2.5, label=f'Quadratic fit (turning pt={turning_point:.1f} yrs)')
ax.axvline(turning_point, color='green', lw=2, linestyle='--', label=f'Turning point = {turning_point:.1f}')
ax.set_title('Quadratic Effect of Experience on log(Wage)', fontsize=12, fontweight='bold')
ax.set_xlabel('Years of Experience'); ax.set_ylabel('log(Wage)')
ax.legend()
plt.savefig(os.path.join(output_dir, 'fig_ch06_02_quadratic.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Section 4 — Interaction Terms

### WHY interactions?
The return to education may differ between men and women.
An interaction term allows the slope to vary by group.

### The model
$$lwage = \beta_0 + \beta_1 educ + \delta_0 female + \delta_1 (female \times educ) + u$$

- For men (female=0): slope = β₁
- For women (female=1): slope = β₁ + δ₁

### Step 4.1 — Create the interaction term

In [ ]:
print('educ (first 5):', df['educ'].values[:5])
print('female (first 5):', df['female'].values[:5])
df['female_educ'] = df['female'] * df['educ']
print('female_educ (first 5):', df['female_educ'].values[:5])
print()
print('Note: female_educ = 0 for all male workers, = educ for female workers')

### Step 4.2 — Estimate interaction model manually (matrix OLS)

In [ ]:
y_int = df['lwage'].values
X_int = np.column_stack([np.ones(len(df)), df['educ'].values, df['female'].values, df['female_educ'].values])
beta_int = np.linalg.inv(X_int.T @ X_int) @ (X_int.T @ y_int)
print('Manual OLS (interaction model):')
for lab, b in zip(['Intercept','educ','female','female_educ'], beta_int):
    print(f'  {lab:<14}: {b:.6f}')

### Step 4.3 — Estimate with statsmodels

In [ ]:
res_int = smf.ols('lwage ~ educ + female + female_educ', data=df).fit()
print('statsmodels output:')
print(res_int.summary())

### Step 4.4 — Compare manual vs statsmodels

In [ ]:
print('COMPARISON: interaction model')
for lab, b_m, var in zip(['Intercept','educ','female','female_educ'], beta_int, ['Intercept','educ','female','female_educ']):
    b_sm = res_int.params[var]
    print(f'  {lab:<14}: manual={b_m:.6f}  sm={b_sm:.6f}  match={np.isclose(b_m, b_sm)}')

### Step 4.5 — Compute partial effects by gender

In [ ]:
b_educ = res_int.params['educ']
b_fema = res_int.params['female']
b_inter= res_int.params['female_educ']
print('PARTIAL EFFECTS OF EDUCATION:')
print(f'  For MALES   (female=0): d(lwage)/d(educ) = beta_educ = {b_educ:.4f}')
print(f'  For FEMALES (female=1): d(lwage)/d(educ) = beta_educ + delta_1 = {b_educ:.4f} + {b_inter:.4f} = {b_educ+b_inter:.4f}')
print(f'  Difference in return to education (F vs M): {b_inter:.4f}')
print(f'  p-value for interaction: {res_int.pvalues["female_educ"]:.4f}')
print(f'  Significant at 5%? {res_int.pvalues["female_educ"] < 0.05}')

### Step 4.6 — Plot separate regression lines

In [ ]:
x_line = np.linspace(0, 18, 200)
b0 = res_int.params['Intercept']
y_male   = b0 + b_educ*x_line
y_female = b0 + b_fema + (b_educ+b_inter)*x_line
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(df[df['female']==0]['educ'], df[df['female']==0]['lwage'], alpha=0.25, s=15, color='steelblue', label='Male')
ax.scatter(df[df['female']==1]['educ'], df[df['female']==1]['lwage'], alpha=0.25, s=15, color='crimson', label='Female')
ax.plot(x_line, y_male,   'b-',  lw=2.5, label=f'Male: slope={b_educ:.3f}')
ax.plot(x_line, y_female, 'r-',  lw=2.5, label=f'Female: slope={b_educ+b_inter:.3f}')
ax.set_title('Interaction: Different Returns to Education by Gender', fontsize=12, fontweight='bold')
ax.set_xlabel('Years of Education'); ax.set_ylabel('log(Wage)')
ax.legend()
plt.savefig(os.path.join(output_dir, 'fig_ch06_03_interaction.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Section 5 — Adjusted R-squared

### WHY Adj-R²?
R² always increases when you add more regressors — even irrelevant ones.
Adjusted R² penalises for adding variables:
$$\bar{R}^2 = 1 - \frac{SSR/(n-k-1)}{SST/(n-1)}$$

### Step 5.1 — Compute adj-R² manually

In [ ]:
res_full = smf.ols('lwage ~ educ + exper + expersq + tenure + female', data=df).fit()
n   = int(res_full.nobs)
k   = len(res_full.params) - 1
SSR = res_full.ssr
SST = res_full.centered_tss
R2_manual    = 1 - SSR/SST
adjR2_manual = 1 - (SSR/(n-k-1)) / (SST/(n-1))
print(f'n={n}, k={k}')
print(f'SSR = {SSR:.4f}')
print(f'SST = {SST:.4f}')
print(f'R2 manual       = {R2_manual:.6f}')
print(f'Adj-R2 manual   = {adjR2_manual:.6f}')

### Step 5.2 — Compare with statsmodels

In [ ]:
R2_sm    = res_full.rsquared
adjR2_sm = res_full.rsquared_adj
print(f'R2 statsmodels      = {R2_sm:.6f}  match={np.isclose(R2_manual, R2_sm)}')
print(f'Adj-R2 statsmodels  = {adjR2_sm:.6f}  match={np.isclose(adjR2_manual, adjR2_sm)}')

### Step 5.3 — Show R² vs Adj-R² as variables are added

In [ ]:
models_comp = [
    ('lwage ~ educ', 'Model 1: educ only'),
    ('lwage ~ educ + exper', 'Model 2: + exper'),
    ('lwage ~ educ + exper + expersq', 'Model 3: + expersq'),
    ('lwage ~ educ + exper + expersq + tenure', 'Model 4: + tenure'),
    ('lwage ~ educ + exper + expersq + tenure + female', 'Model 5: + female'),
    ('lwage ~ educ + exper + expersq + tenure + female + married', 'Model 6: + married'),
]
print(f'{"Model":<40} {"R2":>8} {"Adj-R2":>10}')
print('-' * 62)
for formula, label in models_comp:
    r = smf.ols(formula, data=df).fit()
    print(f'{label:<40} {r.rsquared:>8.4f} {r.rsquared_adj:>10.4f}')

---
## Section 6 — Prediction

### Example Question
> Predict log(wage) for a new worker: 16 years education, 5 years experience, 2 years tenure, male.

### Step 6.1 — Point prediction manually

In [ ]:
b = res_full.params
x_new = {'Intercept':1, 'educ':16, 'exper':5, 'expersq':25, 'tenure':2, 'female':0}
y_pred_manual = sum(b[k]*v for k,v in x_new.items())
print('Prediction for: educ=16, exper=5, tenure=2, male')
print(f'  Predicted log(wage) = {y_pred_manual:.4f}')
print(f'  Predicted wage      = exp({y_pred_manual:.4f}) = ${np.exp(y_pred_manual):.2f}/hour')

### Step 6.2 — Point prediction with statsmodels

In [ ]:
new_obs = pd.DataFrame({'educ':[16],'exper':[5],'expersq':[25],'tenure':[2],'female':[0]})
y_pred_sm = res_full.predict(new_obs).values[0]
print(f'  Predicted log(wage) (statsmodels) = {y_pred_sm:.4f}')
print(f'  Match? {np.isclose(y_pred_manual, y_pred_sm)}')

### Step 6.3 — Prediction interval

In [ ]:
pred_frame = res_full.get_prediction(new_obs).summary_frame(alpha=0.05)
print('Prediction results from statsmodels:')
print(pred_frame[['mean','obs_ci_lower','obs_ci_upper','mean_ci_lower','mean_ci_upper']].round(4).to_string())
print()
print(f'Point prediction : log(wage) = {pred_frame["mean"].values[0]:.4f}')
print(f'95% CI for mean  : [{pred_frame["mean_ci_lower"].values[0]:.4f}, {pred_frame["mean_ci_upper"].values[0]:.4f}]')
print(f'95% PI (new obs) : [{pred_frame["obs_ci_lower"].values[0]:.4f}, {pred_frame["obs_ci_upper"].values[0]:.4f}]')
print()
print('PI is wider than CI because it includes uncertainty about the individual error term.')

---
## Chapter 6 Summary

| Topic | Key formula | Python |
|-------|------------|--------|
| Standardised coef | beta = b * (sx/sy) | regression on z-scores |
| Log-log elasticity | b = %Δy/%Δx | `smf.ols('ly~lx')` |
| Turning point | x* = -b1/(2*b2) | manual |
| Interaction | slope varies by group | `x1*x2` in formula |
| Adj-R2 | 1-(SSR/(n-k-1))/(SST/(n-1)) | `results.rsquared_adj` |
| Prediction | b0+b1*x1+... | `results.predict(new_obs)` |

### Practice Questions
1. Which variable has the largest standardised coefficient? Does the answer change when you add `married`?
2. At what level of experience do returns peak? Are >50% of workers past that point?
3. Test H₀: the return to education is the same for males and females. What is the p-value?
4. Compare adj-R² for log-log vs log-level models. Which is preferred?
5. Predict wage for a female worker with 12 years education, 10 years experience, 5 years tenure.